# Section 2 — Exceptions

**PCAP-31-03 official section** — 5 exam items, 14% of the exam.

Source: [Python Institute PCAP-31-03 syllabus](https://pythoninstitute.org/pcap-exam-syllabus). Explanations below are grounded in the [official Python documentation](https://docs.python.org/3/) (linked per topic) — read the explanation, then run and tweak the code.

---


## 2.1 Handle errors using Python-defined exceptions

All exceptions in Python inherit from `BaseException`. Almost everything you `except` should be a subclass of `Exception` (its most common subclasses: `ArithmeticError` → `ZeroDivisionError`; `LookupError` → `IndexError`, `KeyError`; `ValueError`; `TypeError`). Because `except` matches a class *or any of its subclasses*, ordering matters: put more specific exceptions **before** more general ones, or the general `except` will "steal" errors meant for a more specific handler and the specific one becomes unreachable dead code — a classic PCAP question.

A `try` statement can have all four clauses, always in this order:
- `try`: the code that might fail.
- `except SomeError as e`: runs only if that error (or a subclass) was raised; `e.args` holds the constructor arguments, and `str(e)` prints them.
- `else`: runs only if the `try` block raised **no** exception at all.
- `finally`: **always** runs, exception or not — even if the `try`/`except` returns or re-raises. Used for cleanup (closing files, releasing locks).

`raise SomeError('message')` throws an exception on purpose. `assert condition, 'message'` raises `AssertionError` if `condition` is falsy — meant for catching programmer bugs, not for handling expected user errors.

Docs: [8. Errors and Exceptions](https://docs.python.org/3/tutorial/errors.html) · [Built-in Exceptions](https://docs.python.org/3/library/exceptions.html)


In [1]:
# Specific except BEFORE general except -- order matters!
try:
    result = 10 / 0
except ZeroDivisionError as e:       # specific: matches first
    print('Specific handler caught:', type(e).__name__, '-', e)
except ArithmeticError as e:         # general: ZeroDivisionError IS an ArithmeticError,
    print('General handler caught:', e)   # but it never reaches here, since the above already matched

# Full try / except / else / finally
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print('Cannot divide by zero.')
        return None
    else:
        print('No error happened, result is', result)   # only runs if try succeeded
        return result
    finally:
        print('This always runs (cleanup).')             # runs no matter what

safe_divide(10, 2)
print('---')
safe_divide(10, 0)

# args and raising your own exception
try:
    raise ValueError('bad value', 42)
except ValueError as e:
    print('args:', e.args)
    print('str(e):', str(e))

# assert: for programmer sanity checks, not user input validation
def set_age(age):
    assert age >= 0, 'age cannot be negative'
    return age

print(set_age(30))
try:
    set_age(-5)
except AssertionError as e:
    print('AssertionError:', e)


Specific handler caught: ZeroDivisionError - division by zero
No error happened, result is 5.0
This always runs (cleanup).
---
Cannot divide by zero.
This always runs (cleanup).
args: ('bad value', 42)
str(e): ('bad value', 42)
30
AssertionError: age cannot be negative


**Grouped exceptions (`except*` / `ExceptionGroup`, Python 3.11+).** Sometimes more than one unrelated error needs to be raised and handled *together* — for example, several independent subtasks that each fail for a different reason. `ExceptionGroup('message', [exc1, exc2, ...])` bundles multiple exceptions into a single object that can still be raised and caught. You catch pieces of it with `except*` (note the asterisk) instead of plain `except`: each `except* SomeType` clause matches and handles only the exceptions in the group that are instances of `SomeType`, and Python can run *multiple* `except*` clauses for a single group if different parts match different clauses (`except` and `except*` cannot be mixed in the same `try` statement).

This sandbox runs Python 3.10, which can't parse the `except*` syntax at all (it's a hard syntax error, not a runtime one) — so the cell below builds the code as a string and uses `exec()` to attempt it, catching the `SyntaxError` so this notebook still runs cleanly end-to-end. **Copy the printed code into a Python 3.11+ interpreter** (check with `python3 --version`) to actually see it execute.

In [2]:
import sys
print(f'This environment is running Python {sys.version_info.major}.{sys.version_info.minor}')

grouped_exceptions_demo = '''
try:
    raise ExceptionGroup(
        "multiple failures",
        [ValueError("bad value"), TypeError("bad type"), ValueError("another bad value")]
    )
except* ValueError as eg:
    print("Caught ValueError(s):", eg.exceptions)
except* TypeError as eg:
    print("Caught TypeError(s):", eg.exceptions)
'''

try:
    exec(compile(grouped_exceptions_demo, '<grouped_exceptions_demo>', 'exec'))
except SyntaxError:
    print("SyntaxError here: this Python version doesn't support `except*` (needs 3.11+).")
    print("Run this snippet on a 3.11+ interpreter to see it work:\n")
    print(grouped_exceptions_demo)


This environment is running Python 3.10
SyntaxError here: this Python version doesn't support `except*` (needs 3.11+).
Run this snippet on a 3.11+ interpreter to see it work:


try:
    raise ExceptionGroup(
        "multiple failures",
        [ValueError("bad value"), TypeError("bad type"), ValueError("another bad value")]
    )
except* ValueError as eg:
    print("Caught ValueError(s):", eg.exceptions)
except* TypeError as eg:
    print("Caught TypeError(s):", eg.exceptions)



## 2.2 Extend the exceptions hierarchy with self-defined exceptions

You create a custom exception by subclassing `Exception` (directly or through another exception). This lets calling code catch your specific error by name, or catch it generically via its parent class — and lets you attach extra data (like an error code) as attributes. Custom exception class names conventionally end in `Error`.

Docs: [8.6. User-defined Exceptions](https://docs.python.org/3/tutorial/errors.html#user-defined-exceptions)


In [3]:
class InsufficientFundsError(Exception):
    """Raised when a withdrawal exceeds the available balance."""
    def __init__(self, balance, amount):
        message = f'Cannot withdraw {amount}: balance is only {balance}'
        super().__init__(message)   # sets up args/str() like a normal Exception
        self.balance = balance
        self.amount = amount

def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientFundsError(balance, amount)
    return balance - amount

try:
    withdraw(100, 250)
except InsufficientFundsError as e:
    print('Custom exception caught:', e)
    print('Extra data attached:', e.balance, e.amount)

# A custom exception IS-A Exception, so a generic handler still catches it:
try:
    withdraw(50, 999)
except Exception as e:
    print('Caught via generic Exception handler:', type(e).__name__)


Custom exception caught: Cannot withdraw 250: balance is only 100
Extra data attached: 100 250
Caught via generic Exception handler: InsufficientFundsError
